# 01 - Fundamentos de Ensambles, Bagging y Votación

## Curso de Machine Learning: Modelos de Clasificación

En lugar de depender de las predicciones de un único modelo individual, los **Métodos de Ensamble (Ensemble Methods)** combinan las predicciones de múltiples modelos base para construir un estimador global con mayor precisión, robustez y capacidad de generalización.

---

### Contenidos
1. Filosofía de los Métodos de Ensamble y Teorema del Jurado de Condorcet
2. Taxonomía de Ensambles: Bagging vs. Boosting vs. Stacking
3. Muestreo Bootstrap (Bootstrap Aggregating)
4. Muestras Out-Of-Bag (OOB) y Evaluación Gratuita
5. Implementación Práctica de `BaggingClassifier` con Árboles
6. Clasificadores de Votación (`VotingClassifier`): Voto Duro (*Hard*) vs. Blando (*Soft*)
7. Conclusiones


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)
print("Bibliotecas y entorno listos.")


## 1. Filosofía de los Ensambles: La Sabiduría de las Masas

El fundamento probabilístico de los ensambles proviene del **Teorema del Jurado de Condorcet**:
Si tenemos $M$ clasificadores independientes, cada uno con una probabilidad de acierto individual $p > 0.5$, la probabilidad de que la votación mayoritaria del jurado sea correcta tiende a $1$ a medida que el número de jueces $M \to \infty$.

$$\mathbb{P}(\text{Acierto Mayoritario}) = \sum_{k=\lceil M/2 \rceil}^M \binom{M}{k} p^k (1-p)^{M-k}$$

### Condición Indispensable:
Los clasificadores base deben ser **diversos y descorrelacionados**. Si todos los modelos cometen los mismos errores en los mismos datos, combinarlos no aporta ninguna ventaja.


In [ ]:
# Simulacion del Teorema de Condorcet
p_individual = 0.60 # Cada juez tiene 60% de acierto
m_values = np.arange(1, 101, 2) # Cantidad impar de jueces

from scipy.stats import binom
p_mayoritaria = [1.0 - binom.cdf(m // 2, m, p_individual) for m in m_values]

plt.figure(figsize=(10, 4.5))
plt.plot(m_values, p_mayoritaria, color='#2b5c8f', lw=2.5, label='Probabilidad Votacion Mayoritaria')
plt.axhline(p_individual, color='red', linestyle='--', label=f'Clasificador Individual (p={p_individual})')
plt.axhline(1.0, color='gray', linestyle=':')
plt.xlabel('Numero de Clasificadores en el Ensamble (M)', fontweight='bold')
plt.ylabel('Probabilidad de Acierto Global', fontweight='bold')
plt.title('Teorema del Jurado de Condorcet: El Poder de la Combinacion', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()


## 2. Bagging (Bootstrap Aggregating) y Muestras Out-Of-Bag (OOB)

**Bagging** es una técnica diseñada específicamente para **reducir la varianza** de estimadores inestables (como los árboles de decisión profundos):
1. **Bootstrap**: Se generan $B$ réplicas del conjunto de datos mediante muestreo aleatorio uniforme con reemplazo de tamaño $N$.
2. **Entrenamiento Paralelo**: Se ajusta un modelo base independiente en cada réplica.
3. **Agregación**: Se promedian las probabilidades de los $B$ modelos (clasificación blanda) o se toma la clase mayoritaria.

### La Matemática de las Muestras Out-Of-Bag (OOB):
Para un dataset con $N$ observaciones, la probabilidad de que una muestra particular **NO** sea seleccionada en una réplica bootstrap es:
$$\left(1 - \frac{1}{N}\right)^N \xrightarrow{N \to \infty} \frac{1}{e} \approx 0.3679 \quad (\approx 36.8\%)$$

Esto significa que cada árbol se entrena aproximadamente con el $63.2\%$ de las observaciones, dejando el $36.8\%$ restante como un conjunto de prueba natural (**Muestras OOB**). El **OOB Score** calcula el error promedio de cada muestra utilizando únicamente los árboles que no la vieron durante el entrenamiento.


In [ ]:
# Demostracion de BaggingClassifier con evaluacion OOB
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# Arbol individual
single_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
acc_tree = single_tree.score(X_test, y_test)

# Ensamble Bagging con 100 arboles y evaluacion OOB
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=42,
    n_jobs=-1
)
bagging.fit(X_train, y_train)

acc_bagging = bagging.score(X_test, y_test)
oob_score = bagging.oob_score_

print(f"Exactitud Arbol Individual en Test: {acc_tree*100:.2f}%")
print(f"Exactitud Bagging (100 arboles) en Test: {acc_bagging*100:.2f}%")
print(f"Estimacion Interna OOB Score:         {oob_score*100:.2f}%")


## 3. Clasificadores de Votación (`VotingClassifier`)

Cuando disponemos de modelos de naturalezas algorítmicas heterogéneas (por ejemplo, Regresión Logística, KNN y un Árbol), podemos combinarlos directamente con `VotingClassifier`:

- **Hard Voting (`voting='hard'`)**: Selecciona la clase con mayor cantidad de votos discretos.
- **Soft Voting (`voting='soft'`)**: Promedia las probabilidades predichas $\hat{P}(Y=c \mid \mathbf{x})$ de cada modelo y elige la clase con mayor probabilidad media. Suele ser superior al considerar el nivel de certeza de cada clasificador.


In [ ]:
# Creacion de modelos heterogeneos dentro de pipelines
clf1 = Pipeline([('scaler', StandardScaler()), ('logreg', LogisticRegression(random_state=42))])
clf2 = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])
clf3 = DecisionTreeClassifier(max_depth=4, random_state=42)

# Ensamble por votacion suave (Soft Voting)
voting_clf = VotingClassifier(
    estimators=[('lr', clf1), ('knn', clf2), ('dt', clf3)],
    voting='soft'
)

# Evaluacion individual y combinada con validacion cruzada
modelos = {'Regresion Logistica': clf1, 'KNN': clf2, 'Arbol de Decision': clf3, 'Ensamble Voting': voting_clf}

print("Desempeño con Validacion Cruzada (5-Fold CV):")
print("-" * 50)
for nombre, mod in modelos.items():
    scores = cross_val_score(mod, X_train, y_train, cv=5, scoring='accuracy')
    print(f"{nombre:<22}: {scores.mean()*100:.2f}% (+/- {scores.std()*100:.2f}%)")


## 4. Conclusiones

1. **Reducción de Varianza**: Bagging estabiliza modelos de alta varianza promediando múltiples estimadores entrenados en muestras bootstrap.
2. **OOB Score**: Permite evaluar el desempeño del ensamble sin necesidad de reservar un conjunto de validación separado.
3. **Votación Heterogénea**: `VotingClassifier` permite integrar algoritmos conceptualmente distintos en una única predicción consensuada.

En el siguiente cuaderno, **02_Random_Forest_Clasificacion.ipynb**, estudiaremos cómo Leo Breiman perfeccionó Bagging introduciendo la aleatorización de variables en **Random Forest**.
